# Remote Work Scale — Psychometric Item Generation Lab
**Author:** Olga | **Date:** March 2026

**Objective:** Validate two remote-work scales semantically and generate survey items using RAG-augmented LLMs.

| Scale | Definition |
|---|---|
| **Adaptability** | Ability and motivation to change or fit different task, social, or environmental features |
| **Exploratoriness** | Individual differences in the tendency to seek out new experiences and explore ideas, values, emotions, and sensations |

**Pipeline:** Semantic heatmaps → FAISS index → Retrieval validation → Item generation (No-RAG / RAG / Re-ranked RAG) → Quality evaluation

## Setup — API Clients & Shared Imports
Load all API clients once here. All subsequent chunks use these shared objects.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('.env')

import ollama
from google import genai
from google.genai import types as genai_types
from groq import Groq
from openai import OpenAI

gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
groq_client   = Groq(api_key=os.getenv('GROQ_API_KEY'))
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Scale definitions (shared reference)
SCALES = {
    'Adaptability':    'Ability and motivation to change or fit different task, social, or environmental features.',
    'Exploratoriness': 'Individual differences in the tendency to seek out new experiences and explore ideas, values, emotions, and sensations that differ from previous experience or established preferences.'
}

print('✅ Clients initialized.')

*Observations:*

---
# Phase 1 — Semantic Heatmap Analysis
Establish convergent and discriminant validity for the two remote scales against Big 5 and Maladaptive trait frameworks using 768D sentence embeddings.

## 1.1 — Data Loading & Embedding Setup
Load `aipsych_remote.csv` (frameworks: remote, typical, maladaptive) and generate 768D embeddings using `all-mpnet-base-v2`. This model is used for definition-level semantic analysis only; the IPIP retrieval system uses OpenAI embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('aipsych_remote.csv')
model_mpnet = SentenceTransformer('all-mpnet-base-v2', device='cpu')

definitions = df['definition'].tolist()
print('Encoding with MPNet (768D)...')
embs = model_mpnet.encode(definitions, batch_size=8, show_progress_bar=True)

df_mpnet = df[['label', 'framework']].copy()
for i in range(embs.shape[1]):
    df_mpnet[f'dim_{i}'] = embs[:, i]

print(f'✅ Embeddings ready: {embs.shape}')
print(df['framework'].value_counts())

*Observations:*

## 1.2 — Heatmap: Remote vs. Typical (Big 5)
Convergent validity check. We expect Exploratoriness to align with Typical Openness and Adaptability to align most closely with Conscientiousness.

In [ ]:
embed_cols = [c for c in df_mpnet.columns if c.startswith('dim_')]

remote_df  = df_mpnet[df_mpnet['framework'] == 'remote']
typical_df = df_mpnet[df_mpnet['framework'] == 'typical']

remote_labels  = remote_df['label'].values
typical_labels = typical_df['label'].values

sim = cosine_similarity(remote_df[embed_cols].values, typical_df[embed_cols].values)

fig, ax = plt.subplots(figsize=(14, 5), dpi=300)
cax = ax.imshow(sim, aspect='auto', cmap='bwr', vmin=0, vmax=1)
plt.colorbar(cax, label='Cosine Similarity')
ax.set_xticks(np.arange(len(typical_labels)))
ax.set_xticklabels(typical_labels, rotation=45, ha='right', fontsize=10)
ax.set_yticks(np.arange(len(remote_labels)))
ax.set_yticklabels(remote_labels, fontsize=12)
ax.set_title('Convergent Validity: Remote Scales vs. Big 5', fontsize=16, pad=20)
ax.set_xlabel('Big 5 Traits', fontsize=12)
ax.set_ylabel('Remote Framework Scales', fontsize=12)
plt.tight_layout()
plt.savefig('remote_vs_typical_heatmap.jpeg', format='jpeg', dpi=300)
plt.show()

*Observations:*

## 1.3 — Heatmap: Remote vs. Maladaptive
Discriminant validity check. Remote scales should show low similarity with maladaptive dark traits to confirm they measure distinct, adaptive constructs.

In [ ]:
maladaptive_df = df_mpnet[df_mpnet['framework'] == 'maladaptive']
maladaptive_labels = maladaptive_df['label'].values

sim_m = cosine_similarity(remote_df[embed_cols].values, maladaptive_df[embed_cols].values)

fig, ax = plt.subplots(figsize=(16, 5), dpi=300)
cax = ax.imshow(sim_m, aspect='auto', cmap='bwr', vmin=0, vmax=1)
plt.colorbar(cax, label='Cosine Similarity')
ax.set_xticks(np.arange(len(maladaptive_labels)))
ax.set_xticklabels(maladaptive_labels, rotation=45, ha='right', fontsize=10)
ax.set_yticks(np.arange(len(remote_labels)))
ax.set_yticklabels(remote_labels, fontsize=12)
ax.set_title('Discriminant Validity: Remote Scales vs. Maladaptive Traits', fontsize=16, pad=20)
ax.set_xlabel('Maladaptive Traits', fontsize=12)
ax.set_ylabel('Remote Framework Scales', fontsize=12)
plt.tight_layout()
plt.savefig('remote_vs_maladaptive_heatmap.jpeg', format='jpeg', dpi=300)
plt.show()

*Observations:*

---
# Phase 2 — IPIP Embeddings & FAISS Index
Build a semantic search index over 3,805 validated IPIP personality items. This index is the retrieval backbone for all RAG conditions.

## 2.1 — IPIP Data Loading & OpenAI Embedding
Embed all 3,805 items using `text-embedding-3-small` (1536D) in batches of 100. This takes ~30–40 seconds and costs approximately $0.0006.

In [ ]:
from tqdm import tqdm

items_df = pd.read_csv('TedoneItemAssignmentTable30APR21.csv')
print(f'Loaded {len(items_df)} IPIP items')
print(items_df.head(3))

def get_embeddings_batch(texts):
    response = openai_client.embeddings.create(
        model='text-embedding-3-small',
        input=texts
    )
    return [item.embedding for item in response.data]

embeddings = []
batch_size = 100

print(f'Embedding {len(items_df)} items in batches of {batch_size}...')
for i in tqdm(range(0, len(items_df), batch_size)):
    batch_texts = items_df['text'][i:i+batch_size].tolist()
    try:
        batch_embs = get_embeddings_batch(batch_texts)
        embeddings.extend(batch_embs)
    except Exception as e:
        print(f'\nError on batch {i}: {e}')
        embeddings.extend([None] * len(batch_texts))

items_df['embedding'] = embeddings
print(f'\n✅ Success: {sum(1 for e in embeddings if e is not None)}/{len(items_df)}')

*Observations:*

## 2.2 — FAISS Index Construction
Convert the embedding list to a float32 numpy matrix and build a FAISS `IndexFlatL2` for exact nearest-neighbor search.

In [ ]:
import faiss

# Filter out any failed embeddings
valid_mask = items_df['embedding'].apply(lambda x: x is not None)
items_df_valid = items_df[valid_mask].reset_index(drop=True)

embedding_matrix = np.array(items_df_valid['embedding'].tolist()).astype('float32')
print(f'Embedding matrix shape: {embedding_matrix.shape}')

dimension = embedding_matrix.shape[1]  # 1536
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print(f'✅ FAISS index built with {index.ntotal} vectors')

*Observations:*

---
# Phase 3 — Retrieval Validation
Verify that the FAISS index returns semantically relevant items for our scale definitions before using it in generation.

## 3.1 — RAG Retrieval Quality Check
For each scale, retrieve the top 20 most similar IPIP items. Display results with label breakdown and compute precision, recall, and F1 against a set of expected relevant labels.

In [ ]:
def get_embedding(text):
    """Embed a single query string using text-embedding-3-small."""
    response = openai_client.embeddings.create(
        model='text-embedding-3-small',
        input=[text]
    )
    return response.data[0].embedding

def retrieve_items(query_text, k=20):
    """Retrieve top-k most similar IPIP items for a query."""
    query_emb = np.array([get_embedding(query_text)]).astype('float32')
    distances, indices = index.search(query_emb, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        item = items_df_valid.iloc[idx]
        results.append({'label': item['label'], 'text': item['text'], 'distance': round(float(dist), 4)})
    return results

# Expected relevant labels per scale (adjust based on domain knowledge)
EXPECTED_LABELS = {
    'Adaptability':    {'Flexibility', 'Adaptability', 'Openness', 'Resilience'},
    'Exploratoriness': {'Openness', 'Curiosity', 'Intellect', 'Imagination', 'Adventurousness'}
}

fig, axes = plt.subplots(len(SCALES), 3, figsize=(18, 5 * len(SCALES)), dpi=150)

for row_i, (scale_name, scale_desc) in enumerate(SCALES.items()):
    print(f'\n{'='*60}')
    print(f'Scale: {scale_name}')
    print(f'Query: {scale_desc}')
    results = retrieve_items(scale_desc, k=20)
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))

    retrieved_labels = set(results_df['label'].tolist())
    expected = EXPECTED_LABELS.get(scale_name, set())
    tp = len(retrieved_labels & expected)
    precision = tp / len(retrieved_labels) if retrieved_labels else 0
    recall    = tp / len(expected) if expected else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f'\nPrecision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f}')

    ax_metrics, ax_labels, ax_rank = axes[row_i]

    # Chart 1: Metrics
    ax_metrics.bar(['Precision', 'Recall', 'F1'], [precision, recall, f1], color=['steelblue','coral','seagreen'])
    ax_metrics.set_ylim(0, 1)
    ax_metrics.set_title(f'{scale_name}: Retrieval Metrics')

    # Chart 2: Label breakdown
    label_counts = results_df['label'].value_counts()
    ax_labels.barh(label_counts.index, label_counts.values, color='steelblue')
    ax_labels.set_title(f'{scale_name}: Retrieved Label Breakdown')

    # Chart 3: Distance by rank
    ax_rank.plot(range(1, len(results)+1), results_df['distance'], marker='o')
    ax_rank.set_xlabel('Rank')
    ax_rank.set_ylabel('L2 Distance')
    ax_rank.set_title(f'{scale_name}: Distance by Rank')

plt.tight_layout()
plt.show()

*Observations:*

---
# Phase 4 — Item Generation: No-RAG
Each model generates items from the scale description alone, with no retrieved examples. 15 items per scale per model.

**Prompt structure:** Short first-person statements (5–15 words), mix of positive and negative keying, varied phrasing, no scale name mentioned.

## 4.1 — No-RAG: Gemini (`gemini-2.0-flash-lite`)
Uses Google AI Studio API. Temperature = 0.7 via `GenerateContentConfig`.

In [ ]:
N_ITEMS = 15
MODEL_LABEL = 'gemini'
GEMINI_MODEL = 'gemini-2.0-flash-lite'

def build_no_rag_prompt(scale_name, scale_desc, n):
    return (
        f'Generate {n} personality survey items that measure: {scale_name} — {scale_desc}\n\n'
        'Requirements:\n'
        '- Short, clear first-person statements ("I..."), 5–15 words each\n'
        '- Do NOT mention the scale name in the item\n'
        '- Mix positively and negatively keyed items\n'
        '- Vary phrasing across items\n'
        'Return only the items, one per line, no numbering.'
    )

def generate_gemini(prompt):
    res = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.7)
    )
    return res.text.strip()

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} items for {scale_name}...')
    prompt = build_no_rag_prompt(scale_name, scale_desc, N_ITEMS)
    raw = generate_gemini(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'No-RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_no_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_no_rag.csv')
df_out

*Observations:*

## 4.2 — No-RAG: GPT-4o-mini
Uses OpenAI API. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'gpt'
GPT_MODEL = 'gpt-4o-mini'

def generate_gpt(prompt):
    res = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.7
    )
    return res.choices[0].message.content.strip()

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} items for {scale_name}...')
    prompt = build_no_rag_prompt(scale_name, scale_desc, N_ITEMS)
    raw = generate_gpt(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'No-RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_no_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_no_rag.csv')
df_out

*Observations:*

## 4.3 — No-RAG: LLaMA 3.3 70B (Groq)
Uses Groq Cloud API for high-speed inference. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'llama'
LLAMA_MODEL = 'llama-3.3-70b-versatile'

def generate_llama(prompt):
    res = groq_client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.7
    )
    return res.choices[0].message.content.strip()

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} items for {scale_name}...')
    prompt = build_no_rag_prompt(scale_name, scale_desc, N_ITEMS)
    raw = generate_llama(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'No-RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_no_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_no_rag.csv')
df_out

*Observations:*

## 4.4 — No-RAG: DeepSeek-R1 8B (Ollama / Local)
Runs entirely on local M-series Mac. Strips `<think>...</think>` reasoning tags from output. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'deepseek'
DEEPSEEK_MODEL = 'deepseek-r1:8b'

def generate_deepseek(prompt):
    res = ollama.chat(
        model=DEEPSEEK_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.7}
    )
    content = res['message']['content']
    # Strip chain-of-thought reasoning tags
    clean = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
    return clean

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} items for {scale_name}...')
    prompt = build_no_rag_prompt(scale_name, scale_desc, N_ITEMS)
    raw = generate_deepseek(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'No-RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_no_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_no_rag.csv')
df_out

*Observations:*

---
# Phase 5 — Item Generation: RAG
Each model receives 10 semantically similar IPIP items as context before generating. Context is retrieved from the FAISS index built in Phase 2.

## RAG Context Helper (shared)
Retrieve top-k IPIP items for a scale description and format them as a prompt context block.

In [ ]:
def get_rag_context(scale_desc, k=10):
    """Retrieve top-k IPIP items and format as context string."""
    query_emb = np.array([get_embedding(scale_desc)]).astype('float32')
    distances, indices = index.search(query_emb, k)
    context_lines = []
    for idx in indices[0]:
        item = items_df_valid.iloc[idx]
        context_lines.append(f'[{item["label"]}] {item["text"]}')
    return '\n'.join(context_lines)

def build_rag_prompt(scale_name, scale_desc, context, n):
    return (
        f'Based on these validated personality test items as style examples:\n{context}\n\n'
        f'Generate {n} new items that measure: {scale_name} — {scale_desc}\n\n'
        'Requirements:\n'
        '- Short, clear first-person statements ("I..."), 5–15 words each\n'
        '- Do NOT mention the scale name in the item\n'
        '- Mix positively and negatively keyed items\n'
        '- Vary phrasing across items\n'
        'Return only the items, one per line, no numbering.'
    )

print('✅ RAG context helper ready.')

*Observations:*

## 5.1 — RAG: Gemini (`gemini-2.0-flash-lite`)
Context = 10 FAISS-retrieved IPIP items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'gemini'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} RAG items for {scale_name}...')
    context = get_rag_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_gemini(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_rag.csv')
df_out

*Observations:*

## 5.2 — RAG: GPT-4o-mini
Context = 10 FAISS-retrieved IPIP items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'gpt'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} RAG items for {scale_name}...')
    context = get_rag_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_gpt(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_rag.csv')
df_out

*Observations:*

## 5.3 — RAG: LLaMA 3.3 70B (Groq)
Context = 10 FAISS-retrieved IPIP items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'llama'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} RAG items for {scale_name}...')
    context = get_rag_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_llama(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_rag.csv')
df_out

*Observations:*

## 5.4 — RAG: DeepSeek-R1 8B (Ollama / Local)
Context = 10 FAISS-retrieved IPIP items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'deepseek'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} RAG items for {scale_name}...')
    context = get_rag_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_deepseek(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_rag.csv')
df_out

*Observations:*

---
# Phase 6 — Item Generation: Re-Ranked RAG
Improve retrieval quality by first fetching k×10 candidates from FAISS, then re-ranking with a cross-encoder to select the most semantically relevant context items.

## 6.0 — Cross-Encoder Setup (shared)
Load `ms-marco-MiniLM-L-6-v2` cross-encoder and define the `get_reranked_context()` helper. Run once before the generation chunks.

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def get_reranked_context(scale_desc, k=10, overretrieve_factor=10):
    """FAISS retrieves k*overretrieve_factor candidates, cross-encoder re-ranks to top k."""
    query_emb = np.array([get_embedding(scale_desc)]).astype('float32')
    n_candidates = k * overretrieve_factor
    distances, indices = index.search(query_emb, n_candidates)

    candidates = [items_df_valid.iloc[idx]['text'] for idx in indices[0]]
    pairs = [[scale_desc, c] for c in candidates]
    scores = cross_encoder.predict(pairs)

    # Sort by score descending, take top k
    ranked = sorted(zip(scores, indices[0]), key=lambda x: x[0], reverse=True)[:k]
    context_lines = []
    for score, idx in ranked:
        item = items_df_valid.iloc[idx]
        context_lines.append(f'[{item["label"]}] {item["text"]}')
    return '\n'.join(context_lines)

print('✅ Cross-encoder loaded and re-rank helper ready.')

*Observations:*

## 6.1 — Re-Ranked RAG: Gemini (`gemini-2.0-flash-lite`)
FAISS retrieves 100 candidates → cross-encoder selects top 10 → Gemini generates items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'gemini'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} re-ranked RAG items for {scale_name}...')
    context = get_reranked_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_gemini(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'Re-Ranked RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_reranked_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_reranked_rag.csv')
df_out

*Observations:*

## 6.2 — Re-Ranked RAG: GPT-4o-mini
FAISS retrieves 100 candidates → cross-encoder selects top 10 → GPT-4o-mini generates items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'gpt'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} re-ranked RAG items for {scale_name}...')
    context = get_reranked_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_gpt(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'Re-Ranked RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_reranked_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_reranked_rag.csv')
df_out

*Observations:*

## 6.3 — Re-Ranked RAG: LLaMA 3.3 70B (Groq)
FAISS retrieves 100 candidates → cross-encoder selects top 10 → LLaMA generates items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'llama'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} re-ranked RAG items for {scale_name}...')
    context = get_reranked_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_llama(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'Re-Ranked RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_reranked_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_reranked_rag.csv')
df_out

*Observations:*

## 6.4 — Re-Ranked RAG: DeepSeek-R1 8B (Ollama / Local)
FAISS retrieves 100 candidates → cross-encoder selects top 10 → DeepSeek generates items. Temperature = 0.7.

In [ ]:
MODEL_LABEL = 'deepseek'

results = []
for scale_name, scale_desc in SCALES.items():
    print(f'Generating {N_ITEMS} re-ranked RAG items for {scale_name}...')
    context = get_reranked_context(scale_desc, k=10)
    prompt = build_rag_prompt(scale_name, scale_desc, context, N_ITEMS)
    raw = generate_deepseek(prompt)
    items = [line.strip() for line in raw.split('\n') if line.strip()][:N_ITEMS]
    for item in items:
        results.append({'Scale': scale_name, 'Model': MODEL_LABEL, 'Condition': 'Re-Ranked RAG', 'Item_Text': item})
    print(f'  → {len(items)} items saved')

df_out = pd.DataFrame(results)
df_out.to_csv(f'items_{MODEL_LABEL}_reranked_rag.csv', index=False)
print(f'\n✅ Saved to items_{MODEL_LABEL}_reranked_rag.csv')
df_out

*Observations:*

---
# Phase 7 — Master CSV & Quality Evaluation
Merge all per-model outputs into a single dataset, then evaluate item quality by computing cosine similarity between each generated item and its scale definition.

## 7.1 — Merge to Master CSV
Glob all `items_*.csv` files and combine into `master_items.csv` with columns: `Scale`, `Model`, `Condition`, `Item_Text`.

In [ ]:
import glob

all_files = glob.glob('items_*.csv')
print(f'Found {len(all_files)} files: {all_files}')

dfs = [pd.read_csv(f) for f in all_files]
master_df = pd.concat(dfs, ignore_index=True)
master_df.to_csv('master_items.csv', index=False)

print(f'\n✅ master_items.csv saved: {len(master_df)} total items')
print(master_df.groupby(['Scale', 'Model', 'Condition']).size().reset_index(name='count').to_string(index=False))

*Observations:*

## 7.2 — Cosine Similarity Quality Evaluation
Embed each generated item and compute cosine similarity to its scale definition embedding. Higher similarity = better semantic alignment with the construct. Visualized as boxplots grouped by Model × Condition.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Embed scale definitions
scale_def_embs = {name: np.array(get_embedding(desc)) for name, desc in SCALES.items()}

# Embed all generated items in batches
print(f'Embedding {len(master_df)} generated items...')
all_texts = master_df['Item_Text'].tolist()
item_embeddings = []
for i in tqdm(range(0, len(all_texts), 100)):
    batch = all_texts[i:i+100]
    item_embeddings.extend(get_embeddings_batch(batch))

master_df['embedding'] = item_embeddings

# Compute cosine similarity per item vs. its scale definition
def compute_similarity(row):
    item_emb = np.array(row['embedding']).reshape(1, -1)
    def_emb  = scale_def_embs[row['Scale']].reshape(1, -1)
    return float(cos_sim(item_emb, def_emb)[0][0])

master_df['cosine_sim'] = master_df.apply(compute_similarity, axis=1)

# Boxplot: Model × Condition, split by Scale
fig, axes = plt.subplots(1, len(SCALES), figsize=(16, 6), dpi=150, sharey=True)
conditions = ['No-RAG', 'RAG', 'Re-Ranked RAG']
models = ['gemini', 'gpt', 'llama', 'deepseek']
colors = {'No-RAG': '#4c72b0', 'RAG': '#dd8452', 'Re-Ranked RAG': '#55a868'}

for ax, scale_name in zip(axes, SCALES):
    scale_data = master_df[master_df['Scale'] == scale_name]
    plot_data = []
    plot_labels = []
    for model in models:
        for cond in conditions:
            subset = scale_data[(scale_data['Model'] == model) & (scale_data['Condition'] == cond)]['cosine_sim']
            plot_data.append(subset.tolist())
            plot_labels.append(f'{model}\n{cond.replace(" ", chr(10))}')

    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True)
    for i, (patch, cond) in enumerate(zip(bp['boxes'], conditions * len(models))):
        patch.set_facecolor(colors[cond])
        patch.set_alpha(0.7)
    ax.set_title(f'{scale_name}', fontsize=14)
    ax.set_ylabel('Cosine Similarity to Definition', fontsize=10)
    ax.tick_params(axis='x', labelsize=7)

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=colors[c], alpha=0.7, label=c) for c in conditions]
fig.legend(handles=legend_handles, loc='upper right', fontsize=10)
plt.suptitle('Item Quality: Cosine Similarity by Model and Condition', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('quality_comparison.jpeg', format='jpeg', dpi=300, bbox_inches='tight')
plt.show()

print('\nMean similarity by Model × Condition:')
print(master_df.groupby(['Scale','Model','Condition'])['cosine_sim'].mean().round(3).to_string())

*Observations:*